# Supervisor Template - LangGraph + Composio
**Paras's assignment pattern, ready to clone for any of the 10 projects.**

Shape: `START -> supervisor -> [worker_a | worker_b | worker_c] -> supervisor -> ... -> END`
Workers always return to the supervisor. Each worker is scoped to one Composio toolkit.

Once this notebook works end-to-end, copy it to `01_*`, `02_*`, etc. and only change:
1. The state schema (add domain fields like `ticket_id`, `severity`)
2. The 3 worker prompts and toolkits
3. The supervisor prompt (which workers exist, when to call which)


## 1. Setup

Install once: `pip install langgraph langchain-openai composio composio_langgraph python-dotenv langgraph-checkpoint-sqlite`

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(".env")

# Sanity check (existence only - never print values)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing"
assert os.getenv("COMPOSIO_API_KEY"), "COMPOSIO_API_KEY missing - sign up at composio.dev"
print("env OK")


## 2. State schema

The TypedDict is the contract for what flows between nodes. Always include `messages` (chat history) and `next_worker` (router output). Add domain-specific fields per project.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage

class AgentState(TypedDict):
    # Universal
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    # Domain (override in clones)
    user_request: str
    final_output: str


## 3. LLM

Default to gpt-4o-mini for dev (cheap), swap to gpt-4o for harder tasks.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## 4. Composio toolkits per worker

This is THE production pattern: each worker sees only the tools it needs.
Wrong: one worker with 30 tools (tool-pick errors explode past 10).
Right: 3 workers with 3-5 tools each.

Replace the Action enums below with the ones your project needs.
Full list: https://docs.composio.dev/api/actions

In [ ]:
from composio_langgraph import Action, ComposioToolSet, App

toolset = ComposioToolSet()

# WORKER A - example: Gmail
worker_a_tools = toolset.get_tools(actions=[
    Action.GMAIL_FETCH_EMAILS,
    Action.GMAIL_SEND_EMAIL,
    Action.GMAIL_REPLY_TO_EMAIL,
])

# WORKER B - example: Notion
worker_b_tools = toolset.get_tools(actions=[
    Action.NOTION_QUERY_DATABASE,
    Action.NOTION_FETCH_DATA,
])

# WORKER C - example: Slack
worker_c_tools = toolset.get_tools(actions=[
    Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL,
])


## 5. Workers

Each worker is a ReAct agent over its scoped toolkit. The prebuilt `create_react_agent` handles the Thought-Action-Observation loop for you.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def make_worker(tools, role_prompt: str):
    sub_agent = create_react_agent(llm, tools, prompt=role_prompt)

    def worker_fn(state: AgentState) -> dict:
        # Send the latest human request to the sub-agent
        result = sub_agent.invoke({"messages": state["messages"]})
        # Bubble the agent's final message back up
        last = result["messages"][-1]
        return {"messages": [AIMessage(content=last.content, name="worker")]}
    return worker_fn

worker_a = make_worker(worker_a_tools, "You are the Gmail specialist. Use the Gmail tools to fetch, draft, or send.")
worker_b = make_worker(worker_b_tools, "You are the Notion specialist. Use Notion tools to query the knowledge base.")
worker_c = make_worker(worker_c_tools, "You are the Slack specialist. Use Slack tools to post messages.")


## 6. Supervisor

The supervisor is one LLM call that picks the next worker (or DONE).
Output is constrained to a fixed set of strings via the prompt - simple but works.

In [ ]:
SUPERVISOR_PROMPT = '''You are the supervisor. Read the conversation and pick exactly ONE next step.
Respond with one word from this set:
- worker_a   (use when the task needs Gmail)
- worker_b   (use when the task needs Notion knowledge base)
- worker_c   (use when the task needs Slack notification)
- DONE       (when the task is complete and a final answer is in the messages)

Only respond with the single word. No explanation.
'''

def supervisor(state: AgentState) -> dict:
    decision = llm.invoke([
        SystemMessage(content=SUPERVISOR_PROMPT),
        *state["messages"],
    ])
    pick = decision.content.strip().split()[0]   # be lenient about extra text
    return {"next_worker": pick}


## 7. Router

`add_conditional_edges` calls this function to decide where to go after the supervisor.

In [ ]:
def route(state: AgentState) -> Literal["worker_a", "worker_b", "worker_c", "__end__"]:
    nxt = state["next_worker"]
    if nxt == "DONE":
        return "__end__"
    if nxt in ("worker_a", "worker_b", "worker_c"):
        return nxt
    # safety net - if the LLM said something weird, end the graph
    return "__end__"


## 8. Graph

Wire it up. Workers always go back to the supervisor.

In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(AgentState)
g.add_node("supervisor", supervisor)
g.add_node("worker_a", worker_a)
g.add_node("worker_b", worker_b)
g.add_node("worker_c", worker_c)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "worker_a": "worker_a",
    "worker_b": "worker_b",
    "worker_c": "worker_c",
    "__end__": END,
})
g.add_edge("worker_a", "supervisor")
g.add_edge("worker_b", "supervisor")
g.add_edge("worker_c", "supervisor")


## 9. Checkpoint persistence

Persist conversation state to SQLite so the agent can resume across runs and you can audit what happened.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

app = g.compile(checkpointer=checkpointer)


## 10. Visualise the graph (Paras's submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("Mermaid render not available:", e)
    print(app.get_graph().draw_ascii())


## 11. End-to-end run

Replace the user_request with something realistic for your project.

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "demo-run-1"}, "recursion_limit": 25}

result = app.invoke(
    {"messages": [HumanMessage(content="Fetch the latest 3 emails and summarise them.")]},
    config=config,
)

for m in result["messages"]:
    print(f"[{m.type}]", m.content[:200])


## 12. Bonus: human-in-the-loop on destructive actions

Wrap any worker that sends emails / creates tickets / pays out claims with an interrupt so a human approves first.

```python
from langgraph.types import interrupt, Command

def approve_send(state):
    decision = interrupt({"draft": state["messages"][-1].content,
                          "prompt": "Approve send? yes/no"})
    return {"approved": decision == "yes"}

# add as a node before the actual send action
# resume after human review:
# app.invoke(Command(resume="yes"), config=config)
```


## What to change when you clone this for a specific project

1. **AgentState** - add domain fields (e.g. `ticket_id`, `severity`, `draft_reply`).
2. **Composio actions** in worker_a/b/c - swap the Action enums for what you need.
3. **Worker prompts** - tell each specialist what its job is.
4. **SUPERVISOR_PROMPT** - update the worker descriptions and the routing rules.
5. **Initial input** - replace the demo `HumanMessage` with the real trigger (incoming email body, webhook payload, etc.).

Keep the `route()` and graph-wiring code identical across all 10 projects.
